In [1]:
import base64
import hashlib
import os
import sys
from uuid import uuid4

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from dotenv import dotenv_values
from minio import Minio
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

In [2]:
# Processing date (DDMMYYYY). Keep the same date when retrying this batch.
partition_date = "23112026"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 512m pyspark-shell"

spark = (
    SparkSession.builder
    .appName("SkyPoints")
    .master("local[1]")
    .config("spark.sql.shuffle.partitions", "2")

    # Load the S3 connector and Snowflake Spark/JDBC connectors.
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "net.snowflake:spark-snowflake_2.12:3.2.0-spark_3.5,"
        "net.snowflake:snowflake-jdbc:4.0.2",
    )

    # Locate MinIO.
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:19000")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Authenticate to MinIO.
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

26/11/23 07:47:19 WARN Utils: Your hostname, Monisankars-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/11/23 07:47:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/mskbaruah/.ivy2/cache
The jars for the packages stored in: /Users/mskbaruah/.ivy2/jars


:: loading settings :: url = jar:file:/Users/mskbaruah/code/Incubyte%20Take%20home%20assignment/skypoints-pipeline/skypoints/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


org.apache.hadoop#hadoop-aws added as a dependency
net.snowflake#spark-snowflake_2.12 added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df690ecf-5557-4158-9a4a-99e1dfe958cd;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found net.snowflake#spark-snowflake_2.12;3.2.0-spark_3.5 in central
	found net.snowflake#snowflake-jdbc;4.0.2 in central
	found org.apache.parquet#parquet-avro;1.15.2 in central
	found org.apache.parquet#parquet-column;1.15.2 in central
	found org.apache.parquet#parquet-common;1.15.2 in central
	found org.apache.parquet#parquet-format-structures;1.15.2 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.slf4j#slf4j-api;1.7.33 in central
	found org.apache.parquet#parquet-encoding;1.15.2 in central
	

In [3]:
# 1. Read the saved Snowflake credentials.
sf = dotenv_values(
    "/Users/mskbaruah/code/Incubyte Take home assignment/"
    "skypoints-pipeline/.env.local",
    interpolate=False,
)

# 2. Prepare the encrypted private key for the Spark connector.
passphrase = sf.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")

private_key = serialization.load_pem_private_key(
    open(sf["SNOWFLAKE_LOCAL_PRIVATE_KEY_FILE"], "rb").read(),
    password=passphrase.encode() if passphrase else None,
)

key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)


In [4]:
# Contains private-key material; do not display this dictionary.
sf_options = {
    "sfURL": f"{sf['SNOWFLAKE_ACCOUNT']}.snowflakecomputing.com",
    "sfUser": sf["SNOWFLAKE_USER"],
    "sfRole": sf["SNOWFLAKE_ROLE"],
    "sfWarehouse": sf["SNOWFLAKE_WAREHOUSE"],
    "sfDatabase": sf["SNOWFLAKE_DATABASE"],
    "sfSchema": "RAW",
    "pem_private_key": base64.b64encode(key_bytes).decode("ascii"),
}

# Open the SQL connection used for ingestion-control queries.
sf_connection = snowflake.connector.connect(
    account=sf["SNOWFLAKE_ACCOUNT"],
    user=sf["SNOWFLAKE_USER"],
    role=sf["SNOWFLAKE_ROLE"],
    warehouse=sf["SNOWFLAKE_WAREHOUSE"],
    database=sf["SNOWFLAKE_DATABASE"],
    schema="RAW",
    authenticator="SNOWFLAKE_JWT",
    private_key=key_bytes,
    session_parameters={"TIMEZONE": "UTC"},
)

In [5]:
# 3. Verify that Spark can connect to Snowflake.
(
    spark.read
    .format("net.snowflake.spark.snowflake")
    .options(**sf_options)
    .option("query", "SELECT 1 AS CONNECTION_OK")
    .load()
    .show()
)


26/11/23 07:47:31 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,012 ms, timeout: 1,000 ms
26/11/23 07:47:31 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,016 ms, timeout: 1,000 ms
26/11/23 07:47:31 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,011 ms, timeout: 1,000 ms
26/11/23 07:47:31 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email HTTP/1.1
26/11/23 07:47:31 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,020 ms, timeout: 1,000 ms
26/11/23 07:47:31 ERROR RestRequest: Returning null response. Cause: java.net.SocketException: Host is down, request: GET http://169.254.169.254/metadata/identit

+-------------+
|CONNECTION_OK|
+-------------+
|            1|
+-------------+



In [6]:


landing_path = "s3a://skypoints-landing/"

# Read every matching member file into one DataFrame.
members_raw = (
    spark.read
    .option("pathGlobFilter", "members_*.txt")
    .text(landing_path)
    .withColumn("source_file", F.input_file_name())
)

# Read every matching redemption file into another DataFrame.
redemptions_raw = (
    spark.read
    .option("pathGlobFilter", "redemptions_*.json")
    .option("multiLine", True)
    .json(landing_path)
    .withColumn("source_file", F.input_file_name())
)

members_raw.show(5, truncate = False)
redemptions_raw.show(5, truncate = False)

26/11/23 07:47:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------------------------------------------------------------------------------------------------------+----------------------------------------+
|value                                                                                                     |source_file                             |
+----------------------------------------------------------------------------------------------------------+----------------------------------------+
||H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active|s3a://skypoints-landing/members_0001.txt|
||D|Monserrate Johns|1|20230211|20260922|SLV|Olivia Hart|03|LCA|04011995|I                                 |s3a://skypoints-landing/members_0001.txt|
||D|Hector Heller|2|20231002|20260708|BLU|Maya Arden|02|ARE|13061978|A                                     |s3a://skypoints-landing/members_0001.txt|
||D|Keshaun Lang|3|20180619|20260830|SLV|Leah Nolan|02|PRT|07041995|I                               

In [7]:
# 4. Prepare the raw records and their source filenames.
members_to_write = members_raw.select(
    F.col("value").alias("RAW_RECORD"),
    (
        F.col("source_file")
        if "source_file" in members_raw.columns
        else F.input_file_name()
    ).alias("SOURCE_FILE"),
)

payload_columns = [
    name for name in redemptions_raw.columns
    if name != "source_file"
]

redemptions_to_write = redemptions_raw.select(
    F.struct(*payload_columns).alias("RAW_PAYLOAD"),
    (
        F.col("source_file")
        if "source_file" in redemptions_raw.columns
        else F.input_file_name()
    ).alias("SOURCE_FILE"),
)

In [8]:
# Reuse the same prepared data for hashing and subsequent loading.
# Store it on disk to reduce memory pressure.
members_to_write = members_to_write.persist(StorageLevel.DISK_ONLY)
redemptions_to_write = redemptions_to_write.persist(StorageLevel.DISK_ONLY)

batch_feeds = [
    ("MEMBERS_RAW", "RAW_RECORD", members_to_write),
    ("REDEMPTIONS_RAW", "RAW_PAYLOAD", redemptions_to_write),
]

batch_hasher = hashlib.sha256(b"skypoints-batch-v1")
batch_counts = {}

for table_name, payload_column, dataframe in batch_feeds:
    # Include only source content, preserving null values.
    payload = F.to_json(
        F.struct(F.col(payload_column)),
        options={"ignoreNullFields": "false"},
    )

    # Sort compact fingerprints so input row order does not affect the batch.
    record_hashes = (
        dataframe
        .select(F.sha2(payload, 256).alias("record_hash"))
        .orderBy("record_hash")
    )

    feed_hasher = hashlib.sha256()
    record_count = 0

    # Process fingerprints incrementally rather than collecting the whole batch.
    for record in record_hashes.toLocalIterator():
        feed_hasher.update(bytes.fromhex(record["record_hash"]))
        record_count += 1

    # Combine the feed identity, count and fingerprint into the batch fingerprint.
    batch_hasher.update(
        f"{table_name}:{record_count}:".encode("utf-8")
    )
    batch_hasher.update(feed_hasher.digest())

    batch_counts[table_name] = record_count

batch_hash = batch_hasher.hexdigest()

print(f"Batch fingerprint: {batch_hash}")

for table_name, record_count in batch_counts.items():
    print(f"{table_name}: {record_count:,} records")

Batch fingerprint: ebce4b8b53cd1d330c6aaaa2a72a44cde3a7d87cb2befa97d0eacc4cc1e8ba82
MEMBERS_RAW: 10,001 records
REDEMPTIONS_RAW: 10,000 records


In [13]:
# Check whether this batch has already been successfully loaded.
with sf_connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT BATCH_ID, LOADED_AT
        FROM RAW.BATCH_INGESTION_LOG
        WHERE BATCH_HASH = %s
        """,
        (batch_hash,),
    )

    existing_batch = cursor.fetchone()

batch_already_loaded = existing_batch is not None

if batch_already_loaded:
    previous_batch_id, loaded_at = existing_batch

    print("Batch already loaded. Skip ingestion.")
    print(f"Previous batch ID: {previous_batch_id}")
    print(f"Loaded at: {loaded_at} UTC")
else:
    print("Batch fingerprint is not registered.")

Batch already loaded. Skip ingestion.
Previous batch ID: 1d78725fd9dd48118b534cac78b09bf7
Loaded at: 2026-11-23 02:47:37.233000 UTC


In [10]:
buffer_tables = {
    "MEMBERS_RAW": "MEMBERS_LOAD_BUFFER",
    "REDEMPTIONS_RAW": "REDEMPTIONS_LOAD_BUFFER",
}

buffers_ready = False
batch_id = None

if batch_already_loaded:
    print("Batch already loaded. Skipping buffer loading.")

else:
    batch_id = uuid4().hex

    for raw_table, _, dataframe in batch_feeds:
        buffer_table = buffer_tables[raw_table]

        # Add metadata identifying this loading attempt.
        data_to_buffer = (
            dataframe
            .withColumn("BATCH_ID", F.lit(batch_id))
            .withColumn("PARTITION_DATE", F.lit(partition_date))
            .withColumn("INGESTED_AT", F.current_timestamp())

            # This existing column is unused by our batch-based approach.
            .withColumn("FILE_ID", F.lit(None).cast("string"))
        )

        print(f"Loading {buffer_table}...")

        (
            data_to_buffer.write
            .format("net.snowflake.spark.snowflake")
            .options(**sf_options)
            .option("dbtable", buffer_table)
            .option("column_mapping", "name")
            .option("use_json_in_structured_data", "true")
            .mode("append")
            .save()
        )

        # Verify only the rows belonging to this loading attempt.
        with sf_connection.cursor() as cursor:
            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM RAW.{buffer_table}
                WHERE BATCH_ID = %s
                """,
                (batch_id,),
            )

            actual_count = cursor.fetchone()[0]

        expected_count = batch_counts[raw_table]

        if actual_count != expected_count:
            raise RuntimeError(
                f"{buffer_table}: expected {expected_count:,} records, "
                f"but found {actual_count:,}."
            )

        print(f"Verified {actual_count:,} records in {buffer_table}.")

    buffers_ready = True
    print(f"Both buffers are ready. Batch ID: {batch_id}")

Loading MEMBERS_LOAD_BUFFER...


26/11/23 08:09:45 WARN VersionInfoUtils: The AWS SDK for Java 1.x entered maintenance mode starting July 31, 2024 and will reach end of support on December 31, 2025. For more information, see https://aws.amazon.com/blogs/developer/the-aws-sdk-for-java-1-x-is-in-maintenance-mode-effective-july-31-2024/
You can print where on the file system the AWS SDK for Java 1.x core runtime is located by setting the AWS_JAVA_V1_PRINT_LOCATION environment variable or aws.java.v1.printLocation system property to 'true'.
This message can be disabled by setting the AWS_JAVA_V1_DISABLE_DEPRECATION_ANNOUNCEMENT environment variable or aws.java.v1.disableDeprecationAnnouncement system property to 'true'.
The AWS SDK for Java 1.x is being used here:
at java.base/java.lang.Thread.getStackTrace(Thread.java:1619)
at net.snowflake.spark.shaded.amazonaws.util.VersionInfoUtils.printDeprecationAnnouncement(VersionInfoUtils.java:81)
at net.snowflake.spark.shaded.amazonaws.util.VersionInfoUtils.<clinit>(VersionInfoU

Verified 10,001 records in MEMBERS_LOAD_BUFFER.
Loading REDEMPTIONS_LOAD_BUFFER...


Verified 10,000 records in REDEMPTIONS_LOAD_BUFFER.
Both buffers are ready. Batch ID: b102061398834a44a2b6841f95d572da


In [11]:
baseline_verified = False
baseline_checks = {}

if batch_already_loaded:
    print("Batch already registered. No baseline verification needed.")

elif not buffers_ready:
    raise RuntimeError("Load and verify both buffers before continuing.")

else:
    with sf_connection.cursor() as cursor:
        for raw_table, payload_column, _ in batch_feeds:
            buffer_table = buffer_tables[raw_table]

            comparison_query = f"""
                WITH existing_records AS (
                    SELECT
                        {payload_column} AS PAYLOAD,
                        COUNT(*) AS COPIES
                    FROM RAW.{raw_table}
                    GROUP BY {payload_column}
                ),
                buffered_records AS (
                    SELECT
                        {payload_column} AS PAYLOAD,
                        COUNT(*) AS COPIES
                    FROM RAW.{buffer_table}
                    WHERE BATCH_ID = %s
                    GROUP BY {payload_column}
                )
                SELECT COUNT(*)
                FROM existing_records AS existing
                FULL OUTER JOIN buffered_records AS buffered
                    ON EQUAL_NULL(existing.PAYLOAD, buffered.PAYLOAD)
                WHERE COALESCE(existing.COPIES, 0)
                   <> COALESCE(buffered.COPIES, 0)
            """

            # Retain the checks for rechecking during registration.
            baseline_checks[raw_table] = comparison_query

            cursor.execute(comparison_query, (batch_id,))
            differences = cursor.fetchone()[0]

            if differences:
                raise RuntimeError(
                    f"{raw_table}: found {differences:,} content/count "
                    "mismatches. Existing RAW cannot be registered "
                    "as this batch."
                )

            print(f"{raw_table}: existing RAW matches this batch.")

    baseline_verified = True
    print("Baseline verified. The existing load can be registered.")

MEMBERS_RAW: existing RAW matches this batch.
REDEMPTIONS_RAW: existing RAW matches this batch.
Baseline verified. The existing load can be registered.


In [12]:
if batch_already_loaded:
    print("This batch is already registered.")

elif not buffers_ready or not baseline_verified:
    raise RuntimeError(
        "Both buffers must be ready and baseline verification must pass."
    )

else:
    with sf_connection.cursor() as cursor:
        cursor.execute("ALTER SESSION SET LOCK_TIMEOUT = 60")

        try:
            cursor.execute("BEGIN TRANSACTION")

            # Coordinate with any other pipeline attempting publication.
            cursor.execute("""
                UPDATE RAW.INGESTION_LOCK
                SET LOCK_VERSION = LOCK_VERSION + 1
                WHERE LOCK_ID = 1
            """)

            if cursor.rowcount != 1:
                raise RuntimeError(
                    "Expected exactly one ingestion-lock row."
                )

            # Recheck after acquiring the lock.
            cursor.execute(
                """
                SELECT BATCH_ID
                FROM RAW.BATCH_INGESTION_LOG
                WHERE BATCH_HASH = %s
                """,
                (batch_hash,),
            )

            registered_batch = cursor.fetchone()

            if registered_batch is None:
                # Confirm RAW still matches the buffers while holding the lock.
                for raw_table, comparison_query in baseline_checks.items():
                    cursor.execute(comparison_query, (batch_id,))
                    differences = cursor.fetchone()[0]

                    if differences:
                        raise RuntimeError(
                            f"{raw_table}: RAW no longer matches this batch."
                        )

                # Preserve the batch ID already attached to the RAW records.
                cursor.execute("""
                    SELECT BATCH_ID FROM RAW.MEMBERS_RAW
                    UNION
                    SELECT BATCH_ID FROM RAW.REDEMPTIONS_RAW
                """)

                original_batches = cursor.fetchall()

                if len(original_batches) != 1:
                    raise RuntimeError(
                        "Existing RAW contains multiple batch IDs. "
                        "Review them before registering this baseline."
                    )

                original_batch_id = original_batches[0][0]

                cursor.execute(
                    """
                    INSERT INTO RAW.BATCH_INGESTION_LOG (
                        BATCH_HASH,
                        BATCH_ID,
                        MEMBERS_ROW_COUNT,
                        REDEMPTIONS_ROW_COUNT
                    )
                    VALUES (%s, %s, %s, %s)
                    """,
                    (
                        batch_hash,
                        original_batch_id,
                        batch_counts["MEMBERS_RAW"],
                        batch_counts["REDEMPTIONS_RAW"],
                    ),
                )

            # Remove only this attempt's buffer rows.
            for buffer_table in buffer_tables.values():
                cursor.execute(
                    f"""
                    DELETE FROM RAW.{buffer_table}
                    WHERE BATCH_ID = %s
                    """,
                    (batch_id,),
                )

            sf_connection.commit()

        except Exception:
            sf_connection.rollback()
            raise

    batch_already_loaded = True
    buffers_ready = False

    print("Batch registration confirmed.")
    print("This attempt's buffer rows have been cleared.")

Batch registration confirmed.
This attempt's buffer rows have been cleared.
